# 01 data

Step 1 of the guide: load Sparkov, strip PII, sort by card and time, split by time, basic EDA. Each stage below prints what the data looks like after it, so the transformation can be read off the notebook.

In [1]:
import os, sys, glob, shutil
from google.colab import drive, userdata
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/Colab Notebooks/tfm"
tok = userdata.get("GH_TOKEN")
if not os.path.exists("/content/transaction-fm"):
    !git clone -q https://{tok}@github.com/lb603487485/transaction-fm.git /content/transaction-fm
else:
    !cd /content/transaction-fm && git pull -q
%pip install -q datasets xgboost
%cd /content/transaction-fm
sys.path.insert(0, "/content/transaction-fm/src")
os.makedirs(f"{ROOT}/notebooks", exist_ok=True)
for f in glob.glob("notebooks/*.ipynb"):          # new notebooks from the repo appear in Drive
    dst = f"{ROOT}/notebooks/{os.path.basename(f)}"
    if not os.path.exists(dst):
        shutil.copy(f, dst); print("new notebook in Drive:", os.path.basename(f))
import tfm
print("tfm ok")

Mounted at /content/drive
/content/transaction-fm
tfm ok


## How train / val / test are separated, and why they sit in one file

```
time  ---- 2019-01-01 ------------------ 2020-05-22 ------ 2020-06-21 ------ 2020-12-31 ---->
              train (fit everything here)   val (pick ckpt,    test (fraudTest.csv,
                                            thresholds)        scored once at the end)
```

- The split is by time and written into every row as a `split` column. `test_start` is the first timestamp of fraudTest.csv, so test is exactly the Kaggle test file; `val` is the 30 days before it; both boundaries are saved in `01_data.json`.
- One parquet, not three files: a test-period transaction must see the same card's earlier history (which is in the train period), the same way a production scorer would. Test looking back is not leakage.
- The forbidden direction is blocked in code: every fit-style function in steps 2 to 5 (bin edges, vocabulary, merchant top-N, XGBoost) starts with `assert_train_only(df)` and raises if a val or test row is present; history windows only look backward; checkpoints and thresholds are chosen on val; test is evaluated once in step 6.
- Tests: `tests/test_data.py::test_assert_train_only_blocks_val_and_test_rows` and `test_train_rows_all_precede_val_and_test`.

## Stage 0: raw files as downloaded from Kaggle

In [ ]:
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
from tfm.data import load_raw, clean, assign_splits, eda_stats, save_outputs, PII_COLUMNS

raw, test_start = load_raw(f"{ROOT}/data/raw")
print("rows:", len(raw), " columns:", len(raw.columns), " test_start (first row of fraudTest.csv):", test_start)
print("columns:", list(raw.columns))
raw.head(5)

## Stage 1: clean

Drops the 11 PII columns and the index column, replaces `cc_num` with a dense `card_id`, adds `dist_km` (cardholder to merchant, haversine), sorts by card then time.

In [ ]:
df = clean(raw)
print("dropped:", PII_COLUMNS + ["Unnamed: 0", "cc_num", "lat", "long", "merch_lat", "merch_long"])
print("kept:   ", list(df.columns))
print("rows:", len(df), " cards:", df["card_id"].nunique())
df.head(8)

Same card, consecutive rows, so you can see the time order and the gaps:

In [ ]:
df[df["card_id"] == 0].head(10)

## Stage 2: split by time

`test` = rows at or after the first fraudTest.csv timestamp; `val` = the 30 days before it; `train` = everything earlier. No rows dropped, no shuffling.

In [ ]:
df = assign_splits(df, test_start=test_start, val_days=30)
val_start = test_start - pd.Timedelta(days=30)
print("train <", val_start, " <= val <", test_start, " <= test")
summary = df.groupby("split").agg(rows=("ts", "size"), first_ts=("ts", "min"), last_ts=("ts", "max"), fraud_rate=("is_fraud", "mean")).loc[["train", "val", "test"]]
summary

The rows right at the two boundaries, to show the labels switch exactly there:

In [ ]:
around = df.sort_values("ts")
b1 = around[around["ts"] >= val_start].head(2).index.tolist() + around[around["ts"] < val_start].tail(2).index.tolist()
b2 = around[around["ts"] >= test_start].head(2).index.tolist() + around[around["ts"] < test_start].tail(2).index.tolist()
df.loc[sorted(b1 + b2)].sort_values("ts")[["card_id", "ts", "amt", "is_fraud", "split"]]

## Stage 3: EDA numbers the report needs

In [ ]:
import json
stats = eda_stats(df)
print(json.dumps({k: v for k, v in stats.items() if k != "fraud_rate_by_month"}, indent=2))

## Stage 4: save

`ROOT/data/processed/transactions.parquet` (one file, `split` column) and `ROOT/results/01_data.json` (stats plus the two boundary timestamps).

In [ ]:
stats = save_outputs(df, stats, out_dir=f"{ROOT}/data/processed", results_dir=f"{ROOT}/results", test_start=test_start, val_days=30)
print("val_start:", stats["val_start"], " test_start:", stats["test_start"])
back = pd.read_parquet(f"{ROOT}/data/processed/transactions.parquet")
print("parquet rows:", len(back), " columns:", list(back.columns))

## Charts: fraud rate by month, and the gap between a card's consecutive transactions

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
m = pd.Series(stats["fraud_rate_by_month"])
m.plot(ax=ax[0], marker="o")
ax[0].axvline(list(m.index).index(stats["val_start"][:7]), color="grey", ls="--", label="val start")
ax[0].axvline(list(m.index).index(stats["test_start"][:7]), color="red", ls="--", label="test start")
ax[0].set_title("fraud rate by month"); ax[0].legend(); ax[0].tick_params(axis="x", rotation=45)

gaps = df.groupby("card_id")["ts"].diff().dt.total_seconds().div(3600).dropna()
ax[1].hist(gaps.clip(upper=240), bins=60)
ax[1].set_title("hours between consecutive transactions of a card (clipped at 240)")
plt.tight_layout()
plt.savefig(f"{ROOT}/results/01_data_eda.png", dpi=120)
plt.show()

## What to read off the numbers

- `fraud_rate` around 0.5%, and `fraud_rate` per split tells you whether test is harder or easier than train.
- `fraud_burst_share`: share of fraud rows whose previous fraud on the same card was within 24 h. High means frauds come in bursts, so card-level discovery and leakage checks matter.
- `gap_hours` percentiles decide the time-gap bins in step 2.
- `n_merchants` (about 700) decides the merchant top-N cutoff in step 2.

## Push results and this notebook back to GitHub

In [ ]:
import shutil, glob, os
os.makedirs("results/metrics", exist_ok=True); os.makedirs("results/figures", exist_ok=True)
for f in glob.glob(f"{ROOT}/results/*.json"):
    shutil.copy(f, "results/metrics/")
for f in glob.glob(f"{ROOT}/results/*.png"):
    shutil.copy(f, "results/figures/")
for f in glob.glob(f"{ROOT}/notebooks/*.ipynb"):   # outputs are kept so GitHub shows them
    shutil.copy(f, "notebooks/")
!git config user.email "linboyy@hotmail.com" && git config user.name "Bo Lin"
!git add results notebooks && git commit -q -m "results and notebooks from Colab" && git push -q
!git log --oneline | head -1

In [ ]:
drive.flush_and_unmount()